# 🚗 Analyse Statistique Avancée des Accidents de la Route aux États-Unis


## Importer data

In [4]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
import joblib

#Importer data
df= pd.read_csv("us.csv")

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 47 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             500000 non-null  int64  
 1   ID                     500000 non-null  object 
 2   Source                 500000 non-null  object 
 3   Severity               500000 non-null  int64  
 4   Start_Time             500000 non-null  object 
 5   End_Time               500000 non-null  object 
 6   Start_Lat              500000 non-null  float64
 7   Start_Lng              500000 non-null  float64
 8   End_Lat                0 non-null       float64
 9   End_Lng                0 non-null       float64
 10  Distance(mi)           500000 non-null  float64
 11  Description            500000 non-null  object 
 12  Street                 500000 non-null  object 
 13  City                   499978 non-null  object 
 14  County                 500000 non-nu

## Nettoyage

In [ ]:
# Colonnes à supprimer
drop_cols = [
    "Country", "Source", "Turning_Loop",
    "Description", "Street", "Zipcode",
    "Airport_Code", "Timezone",
    "Nautical_Twilight", "Astronomical_Twilight"
]

# Dtypes optimisés (hors colonnes supprimées)
dtype_map = {
    "ID": "string",
    "Severity": "int8",
    "Start_Lat": "float32",
    "Start_Lng": "float32",
    "End_Lat": "float32",
    "End_Lng": "float32",
    "Distance(mi)": "float32",
    "City": "category",
    "County": "category",
    "State": "category",
    "Temperature(F)": "float32",
    "Wind_Chill(F)": "float32",
    "Humidity(%)": "float32",
    "Pressure(in)": "float32",
    "Visibility(mi)": "float32",
    "Wind_Direction": "category",
    "Wind_Speed(mph)": "float32",
    "Precipitation(in)": "float32",
    "Weather_Condition": "category",
    "Sunrise_Sunset": "category",
    "Civil_Twilight": "category",
}

bool_cols = [
    "Amenity","Bump","Crossing","Give_Way","Junction","No_Exit",
    "Railway","Roundabout","Station","Stop","Traffic_Calming","Traffic_Signal"
]

# -------------------------------
# CHARGEMENT DU FICHIER 500K
# -------------------------------
df = pd.read_csv(
    "us.csv",
    dtype=dtype_map,
    parse_dates=["Start_Time", "End_Time", "Weather_Timestamp"],
    low_memory=False
)

# Conversion en bool
for col in bool_cols:
    df[col] = df[col].astype("boolean")

# Suppression colonnes inutiles
df.drop(columns=drop_cols, errors="ignore", inplace=True)

print("Colonnes supprimées :", drop_cols)


In [ ]:
# Conversion Fahrenheit -> Celsius
df["Temperature(C)"] = (df["Temperature(F)"] - 32) * 5/9
df.drop(columns=["Temperature(F)"], inplace=True)
df["Wind_Chill(C)"] = (df["Wind_Chill(F)"] - 32) * 5/9
df.drop(columns=["Wind_Chill(F)"], inplace=True)

In [29]:
# Suppression doublons
df.drop_duplicates(subset=["ID"], inplace=True)
df.reset_index(drop=True, inplace=True)
print("Doublons supprimés. Nouvelle taille :", df.shape)

Doublons supprimés. Nouvelle taille : (500000, 38)


In [ ]:
# Résumé des colonnes
pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.values,
    "Missing Values": df.isna().sum().values
})


,Column,Dtype,Missing Values
0,Unnamed: 0,int64,0
1,ID,string[python],0
2,Severity,int8,0
3,Start_Time,datetime64[ns],0
4,End_Time,datetime64[ns],0
5,Start_Lat,float32,0
6,Start_Lng,float32,0
7,Distance(mi),float32,0
8,City,string[python],0
9,County,category,0


In [26]:
# Forward fill pour Weather_Timestamp
df = df.sort_values("Start_Time")
df["Weather_Timestamp"] = df["Weather_Timestamp"].ffill()

# Colonnes météo à traiter
weather_cols = [
    "Temperature(C)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)",
    "Wind_Chill(F)", "Precipitation(in)"
]

# Remplissage temporaire pour Wind_Chill et Precipitation
df["Wind_Chill(F)"] = df["Wind_Chill(F)"].fillna(0)
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0)

# Imputation avancée pour les autres colonnes météo numériques
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=30, max_depth=6, n_jobs=-1),
    max_iter=5,
    random_state=42
)

# Colonnes à imputer (sauf Wind_Chill et Precipitation qui sont déjà remplis)
cols_to_impute = ["Temperature(C)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)"]

df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

# Optimisation mémoire
for c in cols_to_impute + ["Wind_Chill(F)", "Precipitation(in)"]:
    df[c] = df[c].astype("float32")

print("✅ Nettoyage et imputation terminés pour l'échantillon 500K")


✅ Nettoyage et imputation terminés pour l'échantillon 500K


c:\Users\leila\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [ ]:
# Résumé des colonnes
pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.values,
    "Missing Values": df.isna().sum().values
})

,Column,Dtype,Missing Values
0,Unnamed: 0,int64,0
1,ID,string[python],0
2,Severity,int8,0
3,Start_Time,datetime64[ns],0
4,End_Time,datetime64[ns],0
5,Start_Lat,float32,0
6,Start_Lng,float32,0
7,Distance(mi),float32,0
8,City,string[python],0
9,County,category,0
